# Geometric Median - Facility Location

## Problem description
A supermarket chain wish to build a store in a small town. There are $m \in \mathbb{N}$ houses that need to be served and these are distributed randomly on a 2D plane. The supermarket owners want to minimise the sum of the distances between the new store and the houses.

## Problem formulation

The model variables are the following:
* $a_i \in \mathbb{R}^2$: Location of house $i$ where $i \in \{1, \dots, m\}$.
* $x \in \mathbb{R}^2$: Decision variable representing the location of the new store.
* $z_i \in \mathbb{R}$: Decision variable representing the distance of the new store from house $i$ where $i \in \{1, \dots, m\}$.

The objective is:

$\min f(z) = \sum_{i=1}^m z_i$

subject to:

$||a_i - x||_2 \leq z_i, \quad \forall i \in \{1, \dots, m\}.$

## Required packages

In [ ]:
import random
import gurobipy as gp
import matplotlib.pyplot as plt

## Data generation

The following code generates random house locations. The value for `number_of_houses` can be changed.

In [ ]:
# The value for m
number_of_houses = 10

# Generating points on a unit square
random.seed(2026)
a_values = []
for i in range(number_of_houses):
    a_values.append((random.random(), random.random()))

The following code provides a preview of the data.

In [ ]:
for i in range(5):
    try:
        print(f"House {i+1} located at {a_values[i]}")
    except:
        pass

## Model building and solving

The following code builds and solves the model using the Gurobi API for Python.

Note the following from [Gurobi documentation](https://docs.gurobi.com/projects/optimizer/en/current/concepts/modeling/constraints.html#norm-constraint):
>Regarding the L2 norm, one obvious complication comes from the fact that enforcing it requires a quadratic constraint. If your model only ever bounds the result from above (e.g., $r = ||x|| \leq 1$), then the resulting constraint will be convex. If your model was otherwise convex, the resulting model will be a (convex) QCP. However, if you try to bound the result from below (e.g., $r = ||x|| \geq y$), adding the L2 norm constraint will lead to a non-convex QCP model, which will typically be significantly harder to solve.

Notice also that the following code models $||a_i - x||_2 = z_i$ instead of $||a_i - x||_2 \leq z_i$. Why is this fine to do?

In [ ]:
# Initialise model
model = gp.Model()

# Add variables
x = model.addVars(2, name="x", lb = 0, ub = 1) # Store location variables
y = model.addVars(number_of_houses,2, name="y", lb = -float('inf')) # Free auxilliary variable
z = model.addVars(number_of_houses, name="z") # Distance variables

# Add objective function (minimise is default)
model.setObjective(z.sum())

# Constrain problem
for i in range(number_of_houses):
    model.addConstr(y[i,0] == x[0] - a_values[i][0])
    model.addConstr(y[i,1] == x[1] - a_values[i][1])
    model.addGenConstrNorm(z[i], [y[i,0], y[i,1]], 2.0) # Adds: z_i = sqrt(y_i[0]^2 + y_i[1]^2)

# Optimize model
model.optimize()

Read the output carefully to see what Gurobi is doing. In particular, notice that:
- Gurobi recognises specific constraint forms when we use `addGenConstrNorm`
- Presolve reduces model to exactly $m = $ `number_of_houses` quadratic constraints

## Solution handling

The following code collects and prints the solution.

In [ ]:
x_sol = (x[0].X, x[1].X)
print(f"Coordinates of store on 2D plane: ({x[0].X}, {x[1].X})")

The following code plots the location of the houses and the store on a 2D plane.

In [ ]:
# Split house locations into two lists
houses_x = [a[0] for a in a_values]
houses_y = [a[1] for a in a_values]

# Plot the lines between houses
for i in range(number_of_houses):
    plt.plot([houses_x[i], x_sol[0]], [houses_y[i], x_sol[1]], 'r-')

# Plot the locations
plt.plot(houses_x, houses_y, 'ko')
plt.plot(x_sol[0], x_sol[1], 'bo')

# Show the plot
plt.show()